In [0]:
# TODOS
# 2. Unit Test & Integration Test
# 3. Separate into different notebooks based on layers

import os
from pyspark.sql import DataFrame

# ── Paths ──────────────────────────────────────────────────────────────────

BASE_DIR = '/Volumes/debora_ryan_susheela_hhs/default/upload_volume'
SILVER_PARQUET_DIR = os.path.join(BASE_DIR, "silver", "data")
os.makedirs(SILVER_PARQUET_DIR, exist_ok=True)

In [0]:
def read_parquet(filepath: str) -> DataFrame:
    data_f = spark.read.parquet(filepath)
    return data_f

In [0]:
def write(input_df: DataFrame, out_dir):
    return input_df.write.mode('overwrite').parquet(out_dir)

In [0]:
# Load Silver Layer data for Gold Layer processing
filepath_rate_baseline = os.path.join(SILVER_PARQUET_DIR, 'rate_baseline')
filepath_copay_primary_care = os.path.join(SILVER_PARQUET_DIR, 'copay_primary_care')
filepath_copay_routine_dental = os.path.join(SILVER_PARQUET_DIR, 'copay_routine_dental')
filepath_copay_basic_dental = os.path.join(SILVER_PARQUET_DIR, 'copay_basic_dental')
filepath_copay_emergency_transport = os.path.join(SILVER_PARQUET_DIR, 'copay_emergency_transport')

df_rate_baseline = read_parquet(filepath_rate_baseline)
df_copay_primary_care = read_parquet(filepath_copay_primary_care)
df_copay_routine_dental = read_parquet(filepath_copay_routine_dental)
df_copay_basic_dental = read_parquet(filepath_copay_basic_dental)
df_copay_emergency_transport = read_parquet(filepath_copay_emergency_transport)

In [0]:

def join_rate_and_copay(df_rates: DataFrame, df_copay: DataFrame) -> DataFrame:
    return (
        df_rates
        .join(df_copay, on="PlanId", how="inner")
        # .dropDuplicates(["PlanId", "IndividualRate", "copay_dollar"])
        .select("PlanId", "IndividualRate", "copay_dollar", "CopayInnTier1")
    )

df_analysis_primary_care = join_rate_and_copay(df_rate_baseline, df_copay_primary_care)
df_analysis_routine_dental = join_rate_and_copay(df_rate_baseline, df_copay_routine_dental)
df_analysis_basic_dental = join_rate_and_copay(df_rate_baseline, df_copay_basic_dental)
df_analysis_emergency_transport = join_rate_and_copay(df_rate_baseline, df_copay_emergency_transport)

print(f"Step 4 — joined analysis rows: {df_analysis_primary_care.count()}")
print(f"Step 4 — joined analysis rows: {df_analysis_routine_dental.count()}")
print(f"Step 4 — joined analysis rows: {df_analysis_basic_dental.count()}")
print(f"Step 4 — joined analysis rows: {df_analysis_emergency_transport.count()}")

In [0]:
from pyspark.sql import functions as F

def remove_outliers(df: DataFrame, col: str) -> DataFrame:
    quantiles = df.approxQuantile(col, [0.01, 0.99], 0.01)
    lower, upper = quantiles[0], quantiles[1]
    return df.filter((F.col(col) >= lower) & (F.col(col) <= upper))

df_analysis_no_outliers_primary_care = remove_outliers(df_analysis_primary_care, "IndividualRate")
df_analysis_no_outliers_routine_dental = remove_outliers(df_analysis_routine_dental, "IndividualRate")
df_analysis_no_outliers_basic_dental = remove_outliers(df_analysis_basic_dental, "IndividualRate")
df_analysis_no_outliers_emergency_transport = remove_outliers(df_analysis_emergency_transport, "IndividualRate")    

# Filter data to keep only copay > 0
df_analysis_no_outliers_positive_copay_primary_care = df_analysis_no_outliers_primary_care.filter(F.col("copay_dollar") > 0.0)
                                              
df_analysis_no_outliers_positive_copay_routine_dental = df_analysis_no_outliers_routine_dental.filter(F.col(                                        "copay_dollar") > 0.0)
df_analysis_no_outliers_positive_copay_basic_dental = df_analysis_no_outliers_basic_dental.filter(F.col("copay_dollar") > 0.0)
df_analysis_no_outliers_positive_copay_emergency_transport = df_analysis_no_outliers_emergency_transport.filter(F.col("copay_dollar") > 0.0)

print(f"Step 5 — joined analysis rows: {df_analysis_no_outliers_primary_care.count()}")
print(f"Step 5 — joined analysis rows: {df_analysis_no_outliers_routine_dental.count()}")
print(f"Step 5 — joined analysis rows: {df_analysis_no_outliers_basic_dental.count()}")
print(f"Step 5 — joined analysis rows: {df_analysis_no_outliers_emergency_transport.count()}")


In [0]:
GOLD_PARQUET_DIR = os.path.join(BASE_DIR, "gold", "data")
os.makedirs(GOLD_PARQUET_DIR, exist_ok=True)

write(df_analysis_no_outliers_positive_copay_primary_care, f"{GOLD_PARQUET_DIR}/primary_care")
write(df_analysis_no_outliers_positive_copay_routine_dental, f"{GOLD_PARQUET_DIR}/routine_dental")
write(df_analysis_no_outliers_positive_copay_basic_dental, f"{GOLD_PARQUET_DIR}/basic_dental")
write(df_analysis_no_outliers_positive_copay_emergency_transport, f"{GOLD_PARQUET_DIR}/emergency_transport")

In [0]:
import matplotlib.pyplot as plt
import numpy as np

dfs = {
    "Primary Care": df_analysis_no_outliers_positive_copay_primary_care,
    "Routine Dental": df_analysis_no_outliers_positive_copay_routine_dental,
    "Basic Dental": df_analysis_no_outliers_positive_copay_basic_dental,
    "Emergency Transport": df_analysis_no_outliers_positive_copay_emergency_transport
}

fig, axs = plt.subplots(2, 2, figsize=(18, 12))
axs = axs.flatten()

for i, (service, df) in enumerate(dfs.items()):
    pdf = df.toPandas()
    summary = (
        pdf.groupby("copay_dollar")["IndividualRate"]
        .agg(mean_premium="mean", median_premium="median", plan_count="count")
        .reset_index()
        .sort_values("copay_dollar")
    )
    ax1 = axs[i]
    ax1.plot(summary["copay_dollar"], summary["mean_premium"],
             marker="o", linewidth=2, color="steelblue", label="Mean Premium ($)")
    ax1.plot(summary["copay_dollar"], summary["median_premium"],
             marker="s", linewidth=2, linestyle="--", color="darkorange", label="Median Premium ($)")
    ax1.set_xlabel(f"{service} Copay ($)", fontsize=12)
    ax1.set_ylabel("Individual Monthly Premium ($)", fontsize=12)
    ax1.set_title(f"Copay Tier vs Premium — {service}", fontsize=13)
    ax1.grid(axis="y", linestyle="--", alpha=0.4)

    x = summary["copay_dollar"].values
    y = summary["mean_premium"].values
    if len(x) > 1:
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        ax1.plot(x, p(x), color="red", linewidth=2, linestyle=":", label="Trend Line (Mean Premium)")

    ax2 = ax1.twinx()
    ax2.bar(summary["copay_dollar"], summary["plan_count"],
            width=1.5, alpha=0.15, color="grey", label="Plan count")
    ax2.set_ylabel("Number of Plans", fontsize=10, color="grey")
    ax2.tick_params(axis="y", labelcolor="grey")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10)

plt.tight_layout()
plt.savefig("copay_linechart_multi.png", dpi=150)
plt.show()

# Group by copay tier → mean and median premium per tier
# summary = (
#     pdf.groupby("copay_dollar")["IndividualRate"]
#     .agg(mean_premium="mean", median_premium="median", plan_count="count")
#     .reset_index()
#     .sort_values("copay_dollar")
# )

# fig, ax1 = plt.subplots(figsize=(12, 6))

# # Mean premium line
# ax1.plot(summary["copay_dollar"], summary["mean_premium"],
#          marker="o", linewidth=2, color="steelblue", label="Mean Premium ($)")
# ax1.plot(summary["copay_dollar"], summary["median_premium"],
#          marker="s", linewidth=2, linestyle="--", color="darkorange", label="Median Premium ($)")
# # ax1.set_xlabel(f"{TARGET_SERVICE} Copay ($)", fontsize=12)
# ax1.set_ylabel("Individual Monthly Premium ($)", fontsize=12)
# # ax1.set_title(f"Copay Tier vs Premium — {TARGET_SERVICE}\n(Hypothesis: higher copay = lower premium)",
# #               fontsize=13)
# ax1.legend(fontsize=10)
# ax1.grid(axis="y", linestyle="--", alpha=0.4)

# # Add trend line (linear regression) for mean premium
# x = summary["copay_dollar"].values
# y = summary["mean_premium"].values
# if len(x) > 1:
#     z = np.polyfit(x, y, 1)
#     p = np.poly1d(z)
#     ax1.plot(x, p(x), color="red", linewidth=2, linestyle=":", label="Trend Line (Mean Premium)")

# # Secondary axis: plan count per tier (bar chart context)
# ax2 = ax1.twinx()
# ax2.bar(summary["copay_dollar"], summary["plan_count"],
#         width=1.5, alpha=0.15, color="grey", label="Plan count")
# ax2.set_ylabel("Number of Plans", fontsize=10, color="grey")
# ax2.tick_params(axis="y", labelcolor="grey")

# lines1, labels1 = ax1.get_legend_handles_labels()
# lines2, labels2 = ax2.get_legend_handles_labels()
# ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10)

# plt.tight_layout()
# plt.savefig("copay_linechart.png", dpi=150)
# plt.show()

In [0]:
display(df_benefits.select('BenefitName').distinct())
df_filtered_primary_care = df_benefits.filter(F.col('BenefitName').contains('Routine Dental Services (Adult)')).select('BenefitName')
display(df_filtered_primary_care)

# Unit Tests

In [0]:
# Install a few helpers we prepared for you
%pip uninstall -y databricks_helpers exercise_ev_databricks_unit_tests

# Install the databricks helpers 
# %pip install git+https://github.com/data-derp/databricks_helpers.git@sr/dbr_17.3_lts_testing
%pip install git+https://github.com/data-derp/databricks_helpers.git

# # Install the databricks test cases
# %pip install git+https://github.com/data-derp/exercise_ev_databricks_unit_tests.git@sr/dbr_17.3_lts_testing
%pip install git+https://github.com/data-derp/exercise_ev_databricks_unit_tests.git

In [0]:
from exercise_ev_databricks_unit_tests.batch_processing_bronze import test_write_e2e

test_write_e2e(dbutils.fs.ls(f"{BASE_DIR}/bronze_output"), spark, display)